<a href="https://colab.research.google.com/github/lautarodibartolo-ae/t8001-pre-procesado-de-datos/blob/main/clase-2-captura-y-variables/02_captura_y_variables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>


# Clase 2 — Los ejemplos

**Taller T8001 · Pre-procesado de datos**

Este notebook acompaña al apunte de la clase 2. No agrega temas nuevos: toma las ideas del apunte
que se entienden mejor viéndolas correr y las muestra funcionando, sobre el relevamiento de doce
hogares que ya conocés. Cada bloque dice a qué sección del apunte corresponde.

Se lee el apunte primero y se ejecuta esto después.


## Cómo se usa

Antes de tocar nada: `Archivo` → `Guardar una copia en Drive`. Si no, los cambios se pierden.

**Para ejecutar una celda: hacé clic adentro y apretá `Shift + Enter`.** El cursor pasa solo a la
siguiente.

Tres reglas:

1. Se ejecuta **en orden, de arriba hacia abajo**. Cada celda usa lo que dejaron las anteriores.
2. Si algo da un resultado raro, casi siempre es por haber ejecutado en desorden. Se arregla con
   `Entorno de ejecución` → `Reiniciar y ejecutar todo`.
3. **Una celda de este notebook falla a propósito.** Está avisada. Cuando llegues, seguí de largo.

Ojo con una consecuencia de las dos últimas: `Reiniciar y ejecutar todo` se frena en la celda que
falla. Después de usarlo, seguí a mano desde la celda siguiente.

Hay cuatro celdas marcadas con **Probá esto**. Son un cambio de una línea para ver qué pasa.


### Empezá por esta

Ejecutala con `Shift + Enter`. Si aparece un resultado abajo, ya está todo listo.


In [ ]:
print("Todo funciona.")
12 * 8


Dos cosas pasaron ahí. `print(...)` muestra lo que le pongas adentro. Y `12 * 8`, la última línea de
la celda, se muestra sola: el notebook siempre muestra el resultado de la última línea.


### Las herramientas

`pandas` es la librería que sabe abrir archivos de datos y trabajar con tablas. Es la única que
hace falta aprender para los datos, y ya la usaste en la clase 1.


In [ ]:
import pandas as pd
from pathlib import Path

Path("crudo").mkdir(exist_ok=True)
Path("generado").mkdir(exist_ok=True)

print("pandas", pd.__version__)


`Path(...).mkdir(exist_ok=True)` crea una carpeta, y `exist_ok` le dice que no se queje si ya
existe. Esas dos carpetas son la **regla del dato crudo intacto** de la clase 1, sección 3.8. En
`crudo/` va lo que llega de la fuente y solo se lee. En `generado/` va todo lo que produce el
notebook, y se puede borrar entera porque se reconstruye sola.


---
# 1. El relevamiento de esta clase

*Apunte, secciones 1.3 y 3.5.*

En vez de bajar un archivo de internet, lo fabricamos acá. Así el notebook funciona siempre, y
además vemos el archivo tal como se escribió, antes de que ningún programa lo interprete.


In [ ]:
tabla = """id,localidad,personas,fecha_visita,ingreso,cobertura_salud,satisfaccion,servicios
1,Ramallo,3,2025-03-14,185000,si,buena,agua;luz;gas
2,Córdoba,5,2025-03-15,240500,no,regular,agua;luz
3,Concepción,2,2025-03-15,,NULL,NULL,luz
4,Ramallo,4,2025-03-16,198000,si,buena,agua;luz;gas
5,ramallo,1,2025-03-16,120000,NULL,NULL,luz
7,Córdoba,6,2025-03-17,310000,no,mala,agua;luz;gas;cloacas
8,Concepción,3,2025-03-17,175000,si,regular,agua;luz
9,Ramallo ,2,2025-03-18,999999,no,mala,luz;gas
10,Córdoba,4,2025-03-19,205000,NULL,NULL,agua;luz;gas
11,Concepción,7,2025-03-20,260000,si,buena,agua;luz;gas;cloacas
13,Ramallo,3,2025-03-20,190000,NULL,NULL,agua;luz
14,Córdoba,2,2025-03-21,168000,no,regular,agua
"""

Path("crudo/relevamiento.csv").write_text(tabla, encoding="utf-8")
print("archivo escrito")


Ahora lo miramos con los ojos, línea por línea. Un CSV es texto plano.


In [ ]:
print(Path("crudo/relevamiento.csv").read_text(encoding="utf-8"))


Tres cosas de esa salida, que después vamos a buscar con código:

- **La línea del hogar 3 tiene dos comas seguidas.** Ahí va el ingreso, y está vacío de verdad:
  nadie escribió nada.
- **Donde dice `NULL` alguien escribió esas cuatro letras.** No es un lugar vacío: es un texto.
- **El hogar 9 dice `Ramallo ` con un espacio antes de la coma.** En la pantalla no se ve.


Recién ahora se lo damos a pandas.


In [ ]:
df = pd.read_csv("crudo/relevamiento.csv")
df


Comparalo con el texto de arriba, porque pandas cambió dos cosas sin preguntar.

Donde el archivo decía `NULL`, ahora dice `NaN`, que es como pandas escribe un faltante: reconoció
esas cuatro letras como una ausencia y las convirtió. Y donde el archivo no decía nada, en el
ingreso del hogar 3, también dice `NaN`. **Dos cosas distintas en el archivo llegaron iguales a la
tabla**, y esa confusión es el tema del bloque 6.


---
# 2. Las cuatro formas de traer un dato

*Apunte, secciones 5.1 a 5.3.*

Cuatro fuentes distintas, una sola librería. Fabricamos cada archivo y lo leemos.


**Forma 1: un CSV.** Ya lo hiciste arriba con una ruta de archivo. La misma función acepta una
dirección web, y es exactamente la misma línea.

> **Esta es la única celda del notebook que necesita internet.** Si no hay conexión vas a ver un
> bloque rojo que dice `HTTPError`. **Ese no es el error a propósito**, que viene mucho más abajo:
> seguí con la celda siguiente, porque ninguna otra depende de esta.


In [ ]:
direccion = "https://raw.githubusercontent.com/lautarodibartolo-ae/t8001-pre-procesado-de-datos/main/clase-2-captura-y-variables/datos/relevamiento.csv"

desde_la_web = pd.read_csv(direccion)
print("forma:", desde_la_web.shape)
print("mismas columnas:", list(desde_la_web.columns) == list(df.columns))
print("mismos datos:   ", desde_la_web.equals(df))
desde_la_web.head(3)


La misma llamada, y lo único que cambió es lo que va entre paréntesis: antes una ruta de archivo,
ahora una dirección web.

Si `mismos datos` dice `False`, no rompiste nada: quiere decir que la copia publicada todavía no es
la última. Es exactamente lo que dice la sección 5.2 del apunte, que una dirección se vuelve a
evaluar cada vez y puede devolver algo distinto.


**Forma 2: una planilla de Excel.** `read_excel` es el `read_csv` de los archivos de Excel. La
primera línea fabrica la planilla con las tres primeras filas, y la segunda la vuelve a leer.


In [ ]:
df.head(3).to_excel("generado/relevamiento.xlsx", index=False)

pd.read_excel("generado/relevamiento.xlsx")


Tres filas y las ocho columnas, igual que en el CSV. Con una planilla real no sería tan fácil: en la
clase 1 viste que hay que decirle en qué hoja está la tabla y en qué fila empieza el encabezado.


**Forma 3: un JSON.** Es el formato con el que contesta una API. No es una tabla: es una lista de
registros, y cada registro puede traer una lista adentro.


In [ ]:
registros = """[
  {"id": 1, "localidad": "Ramallo", "personas": 3, "servicios": ["agua", "luz", "gas"]},
  {"id": 2, "localidad": "Córdoba", "personas": 5, "servicios": ["agua", "luz"]}
]"""

Path("generado/relevamiento.json").write_text(registros, encoding="utf-8")

desde_json = pd.read_json("generado/relevamiento.json")
print(desde_json)
print()
print("qué hay en la celda de servicios:", desde_json["servicios"][0])


Miralo bien: en la columna `servicios`, **cada celda tiene una lista de Python adentro**. No es un
texto con puntos y comas como en el CSV: es una lista de verdad, con tres elementos.

Eso es lo que quiere decir *semiestructurado*, de la clase 1, sección 3.2. Tiene estructura, pero no
es una tabla, y convertirlo a tabla es una decisión que tomás vos.


**Forma 4: una tabla adentro de una página web.** Fabricamos una página con una tabla, de las que
publica cualquier organismo, y se la damos a `read_html`.


In [ ]:
pagina = """<html><head><meta charset="utf-8"></head><body>
<h1>Hogares relevados por localidad</h1>
<table>
  <tr><th>Localidad</th><th>Hogares</th><th>Porcentaje</th></tr>
  <tr><td>Ramallo</td><td>5</td><td>41,67</td></tr>
  <tr><td>Córdoba</td><td>4</td><td>33,33</td></tr>
  <tr><td>Concepción</td><td>3</td><td>25,00</td></tr>
  <tr><td>TOTAL</td><td>12</td><td>100,00</td></tr>
</table></body></html>"""

Path("generado/pagina.html").write_text(pagina, encoding="utf-8")

tablas = pd.read_html("generado/pagina.html")
print("tablas encontradas en la página:", len(tablas))

web = tablas[0]
web


Acá hay tres problemas en una tabla de cuatro filas, y son los de la sección 5.3 del apunte.

`read_html` devolvió una **lista** de tablas, aunque haya una sola. Por eso hizo falta `tablas[0]`
para elegir la primera. En una página con quince tablas, elegir bien es la mitad del trabajo.

La última fila es un **total**, mezclada con los datos. Nada en la tabla dice que lo sea.

Y el tercero es el peor, porque no avisa. Mirá la columna `Porcentaje`.


In [ ]:
print(web.dtypes)
print()
print("suma de la columna Hogares:", web["Hogares"].sum(), " <- son 12 hogares, no 24")
print("el porcentaje de Ramallo:", web["Porcentaje"][0], " <- en la página decía 41,67")


`Porcentaje` llegó como número entero, así que `dtypes` no se queja de nada. Pero en la página decía
`41,67` y acá vale `4167`: pandas leyó la coma como separador de miles y el número quedó **cien
veces más grande**, en una columna que parece perfectamente sana.

Y la suma de `Hogares` da 24 porque la fila de totales se sumó como si fuera una localidad más.

> Una tabla web es una presentación, no un conjunto de datos. Rescatarla cuesta más que leerla.


**Probá esto:** en la celda de la página, cambiá las cuatro comas por puntos y volvé a ejecutar las
dos celdas. Mirá qué le pasa al tipo de la columna `Porcentaje` y a los valores. Si cambiás solo
una, esa queda bien y las otras tres siguen cien veces más grandes.


---
# 3. El formulario sin diseño

*Apunte, secciones 3.4 y 4.2.*

Cinco vecinos de la misma cuadra contestan un formulario de texto libre. Los cinco contestaron el
mismo día, **cuatro tienen tres personas en el hogar** y **cuatro declararon casi el mismo
ingreso**.

Acá no leemos un archivo: escribimos la tabla a mano. Cada nombre entre comillas es una columna, y
la lista entre corchetes tiene sus cinco valores.

Miremos lo que llega a la base.


In [ ]:
malo = pd.DataFrame({
    "fecha":     ["3/4/25", "04/03/2025", "2025-04-03", "3 de abril", "03/04/2025"],
    "localidad": ["Ramallo", "ramallo", "RAMALLO ", "Ramallo.", "ramallo "],
    "personas":  ["3", "tres", "3 personas", "-1", "3"],
    "ingreso":   ["185.000,50", "185000.5", "$185000", "no contesta", "185000,50"],
})
malo


In [ ]:
print("valores distintos en cada columna:")
print(malo.nunique())
print()
print("tipos:")
print(malo.dtypes)


Cinco respuestas, y **tres de las cuatro columnas tienen cinco valores distintos**: cada vecino
inventó su propio formato.

La cuarta columna es `personas`, con cuatro valores, y ahí está el detalle más incómodo: **los
cuatro vecinos de tres personas lo escribieron de tres formas**, `3`, `tres` y `3 personas`. Alcanzó
con que dos lo escribieran distinto para partir en tres un grupo que era uno solo.

Y todas las columnas son texto, incluso las dos que son números. Ninguno de estos vecinos se
equivocó: **todos contestaron bien.** El formulario es el que está mal.


Ahora tratamos de rescatar la columna `personas`. `to_numeric` convierte a número, y
`errors="coerce"` le dice que lo que no pueda convertir lo deje como faltante.


In [ ]:
print(pd.to_numeric(malo["personas"], errors="coerce"))


La conversión rescata 3 de 5. `tres` y `3 personas` se pierden: la información existía y se destruyó
en la captura.

Pero mirá el `-1`. **Sobrevivió**, porque es un número perfectamente válido. Ninguna herramienta te
va a avisar. Un hogar de `-1` personas solo lo detecta alguien que sepa qué rango es posible, y ese
conocimiento se escribe en el diccionario de variables, que es el bloque que sigue.


---
# 4. El diccionario de variables, a medias

*Apunte, sección 6.5.*

El diccionario tiene dos mitades. La parte que es **dato duro** la saca pandas: el nombre de la
columna, el tipo con el que la leyó, cuántos valores distintos hay y cuántos faltan.

Las tres líneas son las tres preguntas, cada una sobre las ocho columnas de una vez. `df.dtypes` da
el tipo, `df.nunique()` cuenta los valores distintos y `df.isna().sum()` cuenta los faltantes.


In [ ]:
diccionario = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "valores_distintos": df.nunique(),
    "faltantes": df.isna().sum(),
})
# rename_axis devuelve una copia: en pandas 2 este índice está compartido con df.columns
diccionario = diccionario.rename_axis("variable")
diccionario


Esa tabla ya contesta dos preguntas. `valores_distintos` dice si una columna es una categoría:
`satisfaccion` tiene 3 y `cobertura_salud` tiene 2, así que lo son; `id` tiene 12 en 12 filas, así
que es un identificador. `faltantes` marca dónde hay que explicar algo.

Lo que pandas **no** puede saber es la escala. Esa mitad la ponemos nosotros.


In [ ]:
# La escala no está en el archivo: es una decisión, y se toma con la tabla del apunte, sección 6.2.
diccionario["escala"] = {
    "id":              "identificador",   # es una etiqueta, no una de las cuatro escalas
    "localidad":       "nominal",         # categorías sin orden
    "personas":        "razon",           # se cuenta, y las proporciones tienen sentido
    "fecha_visita":    "intervalo",       # se resta, no se divide
    "ingreso":         "razon",           # cero pesos es no tener plata
    "cobertura_salud": "nominal",         # dos categorías, sin orden
    "satisfaccion":    "ordinal",         # mala < regular < buena
    "servicios":       "nominal",         # la escala de cada valor de la lista
}

diccionario.to_csv("generado/diccionario_variables.csv")
diccionario[["tipo_pandas", "escala"]]


Mirá las dos últimas filas de esa tabla, porque son la idea entera del bloque.

`satisfaccion` y `servicios` tienen **el mismo tipo de pandas**, que es texto. Y tienen **escalas
distintas**: una es ordinal y la otra nominal. El tipo dice cómo lo guardó la computadora; la escala
dice qué tenés derecho a calcular, y esa no está en el archivo.

Y ojo con dos cosas de lo que acabás de guardar en `generado/diccionario_variables.csv`.

`identificador` no es una de las cuatro escalas del apunte: es la marca de que esa columna no se
analiza. Y el archivo tiene cinco columnas, cuando el apunte pide ocho: `tipo_pandas` no es el
`tipo` del apunte, que dice cualitativa o cuantitativa, y faltan `descripcion`, `unidad`,
`valores_admitidos`, `codigo_faltante` y `origen`, que son justamente las que no salen del archivo.
Esto es el esqueleto del diccionario, no el diccionario.


---
# 5. Los seis pasos del perfilado

*Apunte, sección 7.1.*

La misma rutina, siempre en el mismo orden, sobre cualquier conjunto de datos. Sale de las cinco
preguntas de la clase 1, con dos pasos nuevos: el resumen y la cardinalidad.


Antes de los seis pasos, lo que se hace siempre: mirar las primeras filas. `head()` muestra cinco, y
es lo primero que se pide cuando la tabla no entra en la pantalla.


In [ ]:
df.head()


Ahora sí, los seis pasos.


In [ ]:
print("1. TAMAÑO:", df.shape)
print()
print("2. TIPOS:")
print(df.dtypes)


`ingreso` volvió como decimal y no como entero, porque el tipo entero de pandas no tiene forma de
escribir un faltante. `fecha_visita` volvió como texto. Y `id` volvió como entero, aunque sea una
etiqueta.

Si donde dice `str` te aparece `object`, es lo mismo: así llamaba pandas al texto en las versiones
anteriores, y es lo que puede aparecer en Colab.


In [ ]:
print("3. RESUMEN:")
df.describe()


Lo primero: `describe()` mostró tres columnas y la tabla tiene ocho. Dejó afuera las cinco de
texto, sin avisar, porque de un texto no hay mínimo ni máximo que calcular.

De esta tabla mirás sobre todo `min` y `max`. El resto es estadística que no hace falta acá.

Con una excepción que sorprende: `mean` de `personas` da 3,5, aunque ningún hogar tenga 3,5
personas. Un promedio no es una observación, es un resumen, y puede caer en un valor que no existe
en la tabla.

Y `max` trae los dos primeros hallazgos del paso 3.

**El máximo de `ingreso` es 999999.** No es un ingreso: es un **valor centinela**, un número que
alguien eligió para decir "no sé". Y cuesta caro, porque la columna sigue siendo numérica y ese
valor se promedia. La celda que sigue lo mide.

**El máximo de `id` es 14 y hay doce filas.** La numeración tiene huecos: dos encuestas se perdieron
o se descartaron. Eso es información sobre el relevamiento, no sobre los hogares.

La celda que sigue usa el filtro que se explica en el bloque 6. Por ahora leela como "las filas
donde el ingreso no es 999999".


In [ ]:
sin_centinela = df[df["ingreso"] != 999999]   # el filtro del bloque 6

print("promedio de ingreso, con el centinela adentro:", round(df["ingreso"].mean()))
print("promedio de ingreso, sin el centinela:        ", round(sin_centinela["ingreso"].mean()))


Un 35% de diferencia, y sin un solo error a la vista.


In [ ]:
print("4. FALTANTES:")
print(df.isna().sum())
print()
print("5. DUPLICADOS:")
print("filas repetidas enteras:", df.duplicated().sum())
print("id repetidos:           ", df["id"].duplicated().sum())


Las dos preguntas del paso 5 son distintas y las dos hay que hacerlas. Que no haya filas idénticas
no garantiza que no haya un hogar cargado dos veces con una diferencia mínima; para eso se revisa el
identificador, que acá está limpio.

Y de los faltantes, guardá el dato: **cuatro y cuatro**, en `cobertura_salud` y en `satisfaccion`.
Volvemos sobre eso en el bloque 6.


In [ ]:
print("6. CARDINALIDAD:")
for columna in ["localidad", "cobertura_salud", "satisfaccion", "servicios"]:
    print(columna, "| valores distintos:", df[columna].nunique())


`localidad` tiene cinco valores distintos y hay tres localidades. Ese número que no cierra es la
señal. Para ver qué pasa hay que mirar los valores uno por uno, y eso lo hace `value_counts()`:
cuenta cuántas veces aparece cada valor distinto, del más frecuente al menos frecuente. La primera y
la última línea de su salida son etiquetas de pandas, no datos.


In [ ]:
print(df["localidad"].value_counts())


Ahí está: `Ramallo` aparece **dos veces en la lista**, con 3 y con 1. No es un error de pandas: son
dos textos distintos que se dibujan igual, porque uno termina con un espacio. La celda que sigue lo
demuestra.


In [ ]:
print("los cinco valores, con comillas para ver los bordes:")
for valor in df["localidad"].unique():
    print("   ", repr(valor))

print()
print("largo de 'Ramallo':", len("Ramallo"), "| largo del otro:", len("Ramallo "))


`repr(...)` muestra el valor con sus comillas, y ahí el espacio final aparece. Y los dos largos, 7 y
8, lo terminan de probar: son dos textos distintos. Es el truco para cazar este defecto, que es de
los más difíciles de ver.


In [ ]:
print("servicios: valores distintos:", df["servicios"].nunique())
print(df["servicios"].value_counts())


Seis valores distintos, pero no hay seis servicios. La celda que sigue cuenta los de verdad.

Dos cosas nuevas ahí. `set()` es una bolsa que no admite repetidos: si le agregás `agua` doce veces,
queda una sola. `split(";")` parte el texto de la celda en pedazos, cortando en cada punto y coma. Y
`.strip()` le saca los espacios de los bordes a cada pedazo, que es el defecto que venimos viendo.


In [ ]:
todos = ";".join(df["servicios"])          # las doce celdas pegadas en un solo texto
sueltos = {s.strip() for s in todos.split(";")}

print("servicios que existen:", sorted(sueltos))
print("cantidad:", len(sueltos))


Cuatro servicios, en seis combinaciones. Esa es la firma de una columna **multivalor**, y rompe la
regla de un valor por celda.

`localidad` y `servicios` se **diagnostican hoy y no se tocan**: se arreglan en la clase 3,
`localidad` en el notebook y `servicios` en el apunte, sección 6.4.


**Probá esto:** agregá `;cloacas` al final de la celda de `servicios` de otro hogar, en la tabla del
bloque 1, y volvé a ejecutar desde ahí. Mirá si cambia el conteo de combinaciones, que depende de a
qué hogar se lo agregues, y cómo **no** cambia nunca la cantidad de servicios.

Acá reescribir `crudo/` es parte del ejemplo, porque el crudo lo fabricamos nosotros. Con un archivo
que llega de una fuente, eso no se hace.


---
## La celda que falla a propósito

*Apunte, sección 6.3.*

`satisfaccion` es ordinal: mala, regular, buena. Tiene orden, así que la tentación es promediarla.

> **Esta celda falla a propósito.** Va a salir un bloque rojo y el notebook se frena ahí. Eso es lo
> que queremos ver. Leé la última línea del error y seguí con la celda de abajo.


In [ ]:
df["satisfaccion"].mean()


La última línea del bloque rojo empieza con `TypeError` y habla de texto, o de no poder convertir a
número. Las palabras exactas cambian según la versión de pandas. Lo que importa es el `TypeError`:

pandas se niega a promediar texto, y en este caso hace bien. Pero **no te está protegiendo
de nada**: si vos codificás las categorías a números, promedia sin decir una palabra, y el número
que sale no significa nada. Lo que corresponde con una ordinal es moda y mediana.


In [ ]:
print("moda:", list(df["satisfaccion"].mode()))
print()
print(df["satisfaccion"].value_counts())


Un empate entre `buena` y `regular`, con tres cada una. Eso sí se puede afirmar.

**Probá esto:** agregá una celda nueva, con el botón `+ Código` de arriba a la izquierda, y escribí
`df["satisfaccion"].map({"mala": 1, "regular": 2, "buena": 3}).mean()`. `map` cambia cada categoría
por el número que le asignás entre llaves. Va a devolver un número, sin ningún error, y ese número
no quiere decir nada. Es la trampa de la sección 6.3.


---
# 6. El caso de los ocho `NULL`

*Apunte, secciones 7.3 a 7.5, y 3.2.*

El paso 4 dijo cuatro y cuatro. Pero **"la misma cantidad" no es lo mismo que "las mismas filas"**:
podrían ser cuatro hogares distintos en cada columna y el conteo daría igual.

Para comprobarlo hace falta quedarse con un grupo de filas. Eso se llama **indexado booleano**, y es
la herramienta que más se usa en el taller. *Booleano* quiere decir de dos valores, verdadero o
falso. Va en tres pasos, los mismos del apunte.


**Paso 1: una comparación no devuelve sí o no, devuelve una columna.**


In [ ]:
comparacion = df["personas"] > 4

print("tipo del filtro:", comparacion.dtype)
print("largo:", len(comparacion), "| filas del df:", df.shape[0])
print("cuántos True:", comparacion.sum())
print()
print(comparacion)


Entran doce filas y salen doce valores de verdadero o falso, uno por fila. Esa columna no es el
resultado que buscás: es el **filtro** con el que lo vas a pedir.


**Paso 2: los corchetes hacen dos cosas distintas.**

Con una lista de nombres adentro, eligen columnas. Con un filtro de `True` y `False`, eligen filas.
Es el mismo símbolo para dos operaciones, y es la confusión más común al empezar.


In [ ]:
print("con una lista de nombres, elige columnas:")
print(df[["localidad", "personas"]].head(3))
print()
print("con el filtro, elige filas:")
df[comparacion]


Tres hogares, los de `id` 2, 7 y 11. Dos cosas de ese resultado:

- **La cantidad de `True` es la cantidad de filas que sobreviven.** Por eso `comparacion.sum()`
  sirve para contar sin filtrar.
- **El filtro saca filas, nunca columnas.** Las ocho columnas quedaron intactas.


Ahora el mismo filtro con `isna()` en lugar de una comparación. Se lee "las filas de `df` donde
falta `cobertura_salud`".


In [ ]:
filtro = df["cobertura_salud"].isna()
sin_cobertura = df[filtro]

print("filas sin cobertura_salud:", sin_cobertura.shape[0])
print("de esas, cuántas tampoco tienen satisfaccion:",
      sin_cobertura["satisfaccion"].isna().sum())


**Cuatro de cuatro.** No es casualidad: es una sola causa. Las dos preguntas se dejaron sin
contestar siempre juntas, nunca una sin la otra. Un faltante al azar no se comporta así.

Si algo agrupa a esos cuatro hogares, tiene que explicar **a quiénes**. Las dos sospechas naturales:
fue en una localidad, o fue en una época.


In [ ]:
print("localidad de los cuatro:")
print(sin_cobertura["localidad"].value_counts())
print()
print("fechas de los cuatro:      ",
      sin_cobertura["fecha_visita"].min(), "->", sin_cobertura["fecha_visita"].max())
print("fechas de todo el archivo: ",
      df["fecha_visita"].min(), "->", df["fecha_visita"].max())


Ojo con la primera salida: `Ramallo` y `ramallo` aparecen en dos filas separadas, y son la misma
localidad. Es el defecto del bloque 5, que todavía no arreglamos y que ya molesta.

**Ninguna de las dos explica nada.** Los cuatro hogares están en las tres localidades, así que
ninguna concentra el problema. Y sus fechas van del 15 al 20 de marzo, mientras el relevamiento
entero va del 14 al 21: cubren casi todo el período, así que no hubo un tramo afectado.

Queda una fuente de evidencia sin usar: **el archivo crudo, leído sin que pandas lo interprete.**
Para eso existe la carpeta `crudo/`, y para eso no se toca nunca.


In [ ]:
sin_interpretar = pd.read_csv("crudo/relevamiento.csv", keep_default_na=False)

print(sin_interpretar["cobertura_salud"].value_counts())
print()
print(sin_interpretar["satisfaccion"].value_counts())
print()
print("la columna ingreso, sin interpretar:")
print(sin_interpretar["ingreso"].tolist())


`keep_default_na=False` le dice a pandas que no interprete nada: que traiga los textos tal como
están escritos. Y ahí cambia todo.

**Las dos columnas no están vacías: dicen el texto `NULL`, cuatro veces cada una.** Eso no es un
lugar vacío: es un código que algún programa escribió a propósito.

Y la comparación cierra el argumento. En la lista de ingresos, el del hogar 3 es `''`: una cadena
sin nada adentro, o sea **realmente vacío**. Dos ausencias distintas en la misma tabla, y una de las
dos fue escrita a propósito.

> Los faltantes no son ruido que hay que sacar. Son un dato sobre el instrumento de captura, y a
> veces son el único que queda.


**Probá esto:** cambiá `keep_default_na=False` por `keep_default_na=True` en la celda de arriba y
volvé a ejecutarla. El `NULL` desaparece del conteo, y con él la única pista que había.


Y hay una última cosa que el filtro permite mostrar, y que hasta acá solo leíste: **el sesgo**.

Los ingresos que quedan, sin el centinela, están todos bien cargados. Pero mirá qué pasa si el
encuestador hubiera visitado solo Córdoba.

Dos piezas nuevas en la celda: `&` junta dos filtros, y quedan las filas que cumplen los dos a la
vez; `notna()` es el inverso de `isna()`, verdadero donde el dato está.


In [ ]:
# se saca el centinela y el hogar sin ingreso: así la única diferencia es a quién se visitó
real = df[(df["ingreso"] != 999999) & df["ingreso"].notna()]
solo_cordoba = real[real["localidad"] == "Córdoba"]

print("ingreso promedio, los diez hogares que declararon:", round(real["ingreso"].mean()))
print("ingreso promedio, solo Córdoba:                   ", round(solo_cordoba["ingreso"].mean()))
print("hogares en ese recorte:", len(solo_cordoba), "de", len(real))
print()
print("¿los cuatro de Córdoba están escritos igual?", real["localidad"].str.strip().eq("Córdoba").sum())


Los cuatro valores de Córdoba son correctos, y el promedio se va más de un 12% para arriba. **Eso es
el sesgo:** no hay un solo dato mal, y la conclusión está mal. Ninguna rutina de perfilado lo
detecta, porque el archivo es consistente consigo mismo. Solo se ve sabiendo cómo se eligieron los
hogares.

La última línea es la precaución del bloque 5, aplicada acá: el filtro compara con el texto exacto
`"Córdoba"`, así que conviene confirmar que los cuatro están escritos igual antes de creerle.


---
# Cierre

Los seis bloques que ejecutaste:

1. El relevamiento, primero como texto crudo y después leído por pandas. Dos ausencias distintas en
   el archivo llegaron iguales a la tabla.
2. Cuatro fuentes, una sola librería. Y una tabla web que devuelve un número cien veces más grande
   sin dar ningún error.
3. Un formulario de texto libre, y cinco vecinos que contestaron bien lo mismo de cinco formas.
4. Las dos mitades del diccionario: la que saca pandas y la que exige criterio.
5. Los seis pasos del perfilado, con el centinela, los huecos del identificador, la localidad
   escrita de tres formas y la columna multivalor.
6. El indexado booleano, el cuatro de cuatro y la vuelta al archivo crudo que encuentra el `NULL`.

Quedan afuera del notebook, porque se entienden leyendo, los siete mecanismos de captura y las
cinco reglas del diseño de la captura. Todo eso está en el apunte.

Lo que produjo el notebook está en `generado/` y se puede borrar entero: se reconstruye ejecutando
de arriba hacia abajo. Y si querés seguir probando, las cuatro celdas marcadas con **Probá esto**
son el mejor lugar para empezar.

> **Apunte de la clase 2:**
> [T8001_Apunte_Captura_y_Variables.pdf](https://github.com/lautarodibartolo-ae/t8001-pre-procesado-de-datos/blob/main/clase-2-captura-y-variables/T8001_Apunte_Captura_y_Variables.pdf)
